# Portkey MCP Gateway 

**Portkey's MCP Gateway** is a centralized proxy between MCP clients (agents) and MCP servers. It handles **authentication, access control, credential injection, and full observability** — so agents authenticate once to Portkey, and the gateway handles the rest.

```
AI Agent / MCP Client
        │
        │  x-portkey-api-key (or OAuth)
        ▼
┌──────────────────────────────┐
│    Portkey MCP Gateway       │
│  ┌────────────────────────┐  │
│  │ Auth & Access Control  │  │
│  │ Credential Injection   │  │
│  │ Tool Provisioning      │  │
│  │ Rate Limiting          │  │
│  │ Logging & Observability│  │
│  └────────────────────────┘  │
└──────┬───────┬───────┬───────┘
       │       │       │
       ▼       ▼       ▼
   Linear   GitHub   Slack   ... (MCP Servers)
```

### What we'll cover

| # | Section | What it shows |
|---|---------|---------------|
| 1 | **Setup** | Install deps, configure API key |
| 2 | **Build an MCP Server** | Create a FastMCP server with tools, resources, prompts |
| 3 | **Register Server with Portkey** | Use REST API to add to MCP Registry |
| 4 | **List Registered Servers** | Query all MCP integrations in your workspace |
| 5 | **Connect via Python MCP SDK** | Use `streamablehttp_client` through the gateway |
| 6 | **List Tools** | Discover available tools on a server |
| 7 | **Call Tools** | Execute tool calls through the gateway |
| 8 | **Client Configs** | Claude Desktop, Cursor, VS Code JSON configs |
| 9 | **Using with AI Gateway** | Combine MCP + LLM routing in one flow |
| 10 | **Access Control & Tool Provisioning** | Enable/disable tools per user |
| 11 | **Observability** | Logs, traces, and monitoring |
| 12 | **Cleanup** | Delete test integrations |

>
> **Prerequisites:** Python 3.9+, [Portkey account](https://app.portkey.ai), an API key with **MCP Invoke** permissions

---
## 1. Setup

Install the required packages:
- `mcp` — Official MCP Python SDK (client)
- `fastmcp` — FastMCP server framework
- `portkey-ai` — Portkey SDK (for AI Gateway integration)
- `httpx` / `requests` — HTTP clients for REST API calls

In [19]:
%pip install mcp fastmcp portkey-ai python-dotenv httpx requests -q

Note: you may need to restart the kernel to use updated packages.


In [20]:
import os
import json
import requests
import asyncio
from dotenv import load_dotenv

load_dotenv()

PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
PORTKEY_BASE_URL = "https://api.portkey.ai/v1"
MCP_GATEWAY_URL = "https://mcp.portkey.ai"  # Portkey's hosted MCP Gateway

if not PORTKEY_API_KEY:
    raise EnvironmentError(
        "PORTKEY_API_KEY not found. "
        "Create a .env file with: PORTKEY_API_KEY=your_key_here\n"
        "Ensure the key has MCP Invoke permissions enabled."
    )

masked = PORTKEY_API_KEY[:8] + "*" * 20
print(f"PORTKEY_API_KEY:  {masked}")
print(f"MCP Gateway URL: {MCP_GATEWAY_URL}")
print(f"API Base URL:    {PORTKEY_BASE_URL}")
print("\nReady!")

PORTKEY_API_KEY:  w1vVZCBe********************
MCP Gateway URL: https://mcp.portkey.ai
API Base URL:    https://api.portkey.ai/v1

Ready!


---
## 2. Build an MCP Server

Before we can test the gateway, we need an MCP server. Below we define a **FastMCP server** with:
- **Tools** — Functions the LLM can call (add numbers, get time, weather, currency)
- **Resources** — Read-only data endpoints (system info)
- **Prompts** — Reusable prompt templates

```
FastMCP Server
├── Tools
│   ├── add_numbers       — Basic arithmetic
│   ├── get_current_time  — Server timestamp
│   ├── get_exchange_rates — Live currency rates
│   └── convert_currency  — Currency conversion
├── Resources
│   └── system_info       — Python version, OS, hostname
└── Prompts
    └── analyze_data      — Data analysis template
```

> **Note:** This cell *defines* the server. To actually run it, execute the start cell (2b) or run `python mcp_demo_server.py` from a terminal.

In [21]:
# ── 2a: Define the MCP Server ─────────────────────────────────────────────────

from fastmcp import FastMCP
import datetime
import platform
import random
import httpx
from typing import Dict

mcp_server = FastMCP("Demo MCP Server")

# ── Tools ─────────────────────────────────────────────────────────────────────

@mcp_server.tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b

@mcp_server.tool
def get_current_time() -> str:
    """Return the current server time as a formatted string."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@mcp_server.tool
def generate_random_number(min_val: int = 1, max_val: int = 100) -> int:
    """Generate a random integer between min_val and max_val (inclusive)."""
    if min_val > max_val:
        raise ValueError("min_val cannot be greater than max_val")
    return random.randint(min_val, max_val)

EXCHANGE_API_KEY = "9e233f662d770df215d4d0fa"  # free tier demo key

@mcp_server.tool
async def get_exchange_rates(base_currency: str) -> dict:
    """Fetch the latest exchange rates for a given base currency (e.g. USD, EUR, INR)."""
    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_API_KEY}/latest/{base_currency.upper()}"
    async with httpx.AsyncClient(timeout=10.0) as client:
        response = await client.get(url)
        response.raise_for_status()
        data = response.json()
    if data.get("result") != "success":
        raise Exception(f"Exchange API error: {data}")
    return {
        "base": data["base_code"],
        "rates": dict(list(data["conversion_rates"].items())[:10]),  # top 10 for brevity
        "last_updated": data["time_last_update_utc"],
    }

@mcp_server.tool
async def convert_currency(from_currency: str, to_currency: str, amount: float) -> dict:
    """Convert an amount from one currency to another using live rates."""
    url = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_API_KEY}/latest/{from_currency.upper()}"
    async with httpx.AsyncClient(timeout=10.0) as client:
        response = await client.get(url)
        response.raise_for_status()
        data = response.json()
    rate = data["conversion_rates"].get(to_currency.upper())
    if not rate:
        raise ValueError(f"Unsupported currency: {to_currency}")
    return {
        "from": from_currency.upper(),
        "to": to_currency.upper(),
        "amount": amount,
        "converted_amount": round(rate * amount, 2),
        "rate": rate,
    }

# ── Resources ─────────────────────────────────────────────────────────────────

@mcp_server.resource(name="system_info", uri="resource://system_info")
def get_system_info() -> dict:
    """Expose system metadata as an MCP resource."""
    return {
        "python_version": platform.python_version(),
        "os": platform.system(),
        "hostname": platform.node(),
    }

# ── Prompts ───────────────────────────────────────────────────────────────────

@mcp_server.prompt
def analyze_data(data: str) -> str:
    """Prompt template for data analysis."""
    return f"Analyze the following data and provide insights:\n\n{data}"

print("MCP Server defined with:")
print("  Tools:     add_numbers, get_current_time, generate_random_number,")
print("             get_exchange_rates, convert_currency")
print("  Resources: system_info")
print("  Prompts:   analyze_data")

MCP Server defined with:
  Tools:     add_numbers, get_current_time, generate_random_number,
             get_exchange_rates, convert_currency
  Resources: system_info
  Prompts:   analyze_data


### 2b. Start the MCP Server

The MCP server needs to be **running** for the gateway to proxy requests to it.

**Option A** — Run from terminal (recommended for demos):
```bash
python -c "
from fastmcp import FastMCP
import datetime, platform, random, httpx

mcp = FastMCP('Demo MCP Server')

@mcp.tool
def add_numbers(a: int, b: int) -> int:
    return a + b

@mcp.tool
def get_current_time() -> str:
    return datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

mcp.run(transport='streamable-http', host='0.0.0.0', port=8000)
"
```

**Option B** — If your server is already deployed or you're using a public MCP server (like DeepWiki), skip this step.

**Option C** — For cloud-hosted Portkey, expose your local server via **ngrok**:
```bash
ngrok http 8000
# Copy the https://xxxx.ngrok.io URL for registration
```

---
## 3. Register MCP Server with Portkey

Add your MCP server to Portkey's **MCP Registry** so the gateway knows how to proxy to it.

### Registration payload

| Field | Description |
|-------|-------------|
| `name` | Human-readable name |
| `url` | Server's MCP endpoint URL |
| `slug` | URL-safe identifier (used in connection URL) |
| `auth_type` | `none`, `api_key`, `oauth`, or `headers` |
| `transport` | `http` (Streamable HTTP) or `sse` |

Once registered, the connection URL becomes:
```
https://mcp.portkey.ai/{slug}/mcp
```

In [4]:
# ── Register a custom MCP server ───────────────────────────────────────────────

# Change this URL to your server's public URL
# For local dev: use ngrok URL (e.g., "https://xxxx.ngrok-free.app/mcp")
# For deployed: use the actual URL
MCP_SERVER_URL = "http://0.0.0.0:8000/mcp"  # local default
MCP_SERVER_SLUG = "demo-mcp-server"

payload = {
    "name": "Demo MCP Server",
    "url": MCP_SERVER_URL,
    "slug": MCP_SERVER_SLUG,
    "auth_type": "none",
    "transport": "http",
}

headers = {
    "x-portkey-api-key": PORTKEY_API_KEY,
    "Content-Type": "application/json",
}

print("Registering MCP server with Portkey...")
print(f"  Name: {payload['name']}")
print(f"  URL:  {payload['url']}")
print(f"  Slug: {payload['slug']}")
print("-" * 60)

try:
    resp = requests.post(
        f"{PORTKEY_BASE_URL}/mcp-integrations",
        json=payload,
        headers=headers,
        timeout=30,
    )
    result = resp.json()
    print(f"Status: {resp.status_code}")
    print(json.dumps(result, indent=2))

    if resp.status_code in (200, 201):
        integration_id = result.get("id") or result.get("data", {}).get("id")
        print(f"\nRegistered successfully!")
        print(f"  Integration ID:  {integration_id}")
        print(f"  Connection URL:  {MCP_GATEWAY_URL}/{MCP_SERVER_SLUG}/mcp")
    else:
        print(f"\nRegistration returned status {resp.status_code}")
except Exception as e:
    print(f"Error: {e}")

Registering MCP server with Portkey...
  Name: Demo MCP Server
  URL:  http://0.0.0.0:8000/mcp
  Slug: demo-mcp-server
------------------------------------------------------------
Status: 200
{
  "id": "70223606-e3d5-42db-96ef-51a7144751c5",
  "slug": "demo-mcp-server",
  "object": "mcp-integration"
}

Registered successfully!
  Integration ID:  70223606-e3d5-42db-96ef-51a7144751c5
  Connection URL:  https://mcp.portkey.ai/demo-mcp-server/mcp


---
## 4. List Registered MCP Servers

Query all MCP integrations in your workspace.

In [5]:
print("Fetching MCP integrations...")
print("-" * 60)

try:
    resp = requests.get(
        f"{PORTKEY_BASE_URL}/mcp-integrations?page_size=100",
        headers={"x-portkey-api-key": PORTKEY_API_KEY},
        timeout=30,
    )
    data = resp.json()
    integrations = data.get("data", data) if isinstance(data, dict) else data

    if isinstance(integrations, list):
        print(f"Found {len(integrations)} integration(s):\n")
        for i, item in enumerate(integrations, 1):
            print(f"  [{i}] {item.get('name', 'N/A')}")
            print(f"       ID:        {item.get('id', 'N/A')}")
            print(f"       Slug:      {item.get('slug', 'N/A')}")
            print(f"       URL:       {item.get('url', 'N/A')}")
            print(f"       Transport: {item.get('transport', 'N/A')}")
            print(f"       Auth:      {item.get('auth_type', 'N/A')}")
            print()
    else:
        print(json.dumps(data, indent=2))
except Exception as e:
    print(f"Error: {e}")

Fetching MCP integrations...
------------------------------------------------------------
Found 3 integration(s):

  [1] Demo MCP Server
       ID:        70223606-e3d5-42db-96ef-51a7144751c5
       Slug:      demo-mcp-server
       URL:       http://0.0.0.0:8000/mcp
       Transport: http
       Auth:      none

  [2] agent-cli-sid-test
       ID:        b7209477-1093-40d0-bd4a-bb2b5a2ca8f3
       Slug:      agent-cli-sid-test
       URL:       https://mcp.deepwiki.com/mcp
       Transport: http
       Auth:      headers

  [3] hk-test-mcp
       ID:        159760ea-431e-4de6-ad3b-941c3813bf85
       Slug:      hk-test-mcp
       URL:       https://nondeluded-kelle-lavishly.ngrok-free.dev/mcp
       Transport: http
       Auth:      none



---
## 5. Connect via Python MCP SDK

Use the official **MCP Python SDK** to connect to any MCP server through the Portkey gateway.

The key insight: your client talks to `mcp.portkey.ai/{slug}/mcp` — the gateway handles auth, logging, and proxying to the actual server.

```
Your Code  ──▶  mcp.portkey.ai/{slug}/mcp  ──▶  Actual MCP Server
               (Portkey Gateway)               (your server / Linear / GitHub / ...)
```

In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Connection URL — points to Portkey gateway, not the server directly
GATEWAY_CONNECTION_URL = f"{MCP_GATEWAY_URL}/{MCP_SERVER_SLUG}/mcp"
GATEWAY_HEADERS = {"x-portkey-api-key": PORTKEY_API_KEY}

print(f"Connecting to: {GATEWAY_CONNECTION_URL}")
print("-" * 60)

async def connect_and_initialize():
    """Connect to an MCP server through the Portkey gateway."""
    async with streamablehttp_client(
        GATEWAY_CONNECTION_URL, headers=GATEWAY_HEADERS
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print("Connected and initialized!")
            
            # Return session info
            tools = await session.list_tools()
            print(f"Server has {len(tools.tools)} tool(s) available.")
            return tools

try:
    tools = await connect_and_initialize()
except Exception as e:
    print(f"Connection error: {e}")
    print("\nMake sure:")
    print("  1. Your MCP server is running and reachable")
    print("  2. The server is registered in Portkey's MCP Registry")
    print("  3. Your API key has MCP Invoke permissions")

Connecting to: https://mcp.portkey.ai/demo-mcp-server/mcp
------------------------------------------------------------
Connection error: unhandled errors in a TaskGroup (1 sub-exception)

Make sure:
  1. Your MCP server is running and reachable
  2. The server is registered in Portkey's MCP Registry
  3. Your API key has MCP Invoke permissions


---
## 6. List Tools

Discover what tools are available on the server. The gateway filters tools based on your **access control** settings — disabled tools won't appear.

In [7]:
async def list_available_tools():
    """List all tools available on the MCP server."""
    async with streamablehttp_client(
        GATEWAY_CONNECTION_URL, headers=GATEWAY_HEADERS
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()

            print(f"Available tools ({len(tools.tools)}):\n")
            for tool in tools.tools:
                print(f"  {tool.name}")
                print(f"    Description: {tool.description}")
                if tool.inputSchema and tool.inputSchema.get("properties"):
                    params = list(tool.inputSchema["properties"].keys())
                    print(f"    Parameters:  {', '.join(params)}")
                print()

try:
    await list_available_tools()
except Exception as e:
    print(f"Error: {e}")

Error: unhandled errors in a TaskGroup (1 sub-exception)


---
## 7. Call Tools

Execute tool calls through the gateway. Every call is **logged** in Portkey with full request/response details.

In [8]:
async def call_tool(tool_name: str, arguments: dict):
    """Call a specific tool through the MCP gateway."""
    async with streamablehttp_client(
        GATEWAY_CONNECTION_URL, headers=GATEWAY_HEADERS
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            return result

# ── Test 7.1: add_numbers ─────────────────────────────────────────────────────
print("7.1 — add_numbers(17, 25)")
print("-" * 40)
try:
    result = await call_tool("add_numbers", {"a": 17, "b": 25})
    print(f"  Result: {result}")
except Exception as e:
    print(f"  Error: {e}")

print()

7.1 — add_numbers(17, 25)
----------------------------------------
  Error: unhandled errors in a TaskGroup (1 sub-exception)



In [9]:
# ── Test 7.2: get_current_time ────────────────────────────────────────────────
print("7.2 — get_current_time()")
print("-" * 40)
try:
    result = await call_tool("get_current_time", {})
    print(f"  Server time: {result}")
except Exception as e:
    print(f"  Error: {e}")

print()

7.2 — get_current_time()
----------------------------------------
  Error: unhandled errors in a TaskGroup (1 sub-exception)



In [10]:
# ── Test 7.3: generate_random_number ──────────────────────────────────────────
print("7.3 — generate_random_number(min_val=1, max_val=1000)")
print("-" * 40)
try:
    result = await call_tool("generate_random_number", {"min_val": 1, "max_val": 1000})
    print(f"  Random number: {result}")
except Exception as e:
    print(f"  Error: {e}")

print()

7.3 — generate_random_number(min_val=1, max_val=1000)
----------------------------------------
  Error: unhandled errors in a TaskGroup (1 sub-exception)



In [11]:
# ── Test 7.4: get_exchange_rates ───────────────────────────────────────────────
print("7.4 — get_exchange_rates(base_currency='USD')")
print("-" * 40)
try:
    result = await call_tool("get_exchange_rates", {"base_currency": "USD"})
    print(f"  Result: {result}")
except Exception as e:
    print(f"  Error: {e}")

print()

7.4 — get_exchange_rates(base_currency='USD')
----------------------------------------
  Error: unhandled errors in a TaskGroup (1 sub-exception)



In [12]:
# ── Test 7.5: convert_currency ─────────────────────────────────────────────────
print("7.5 — convert_currency(USD → INR, amount=100)")
print("-" * 40)
try:
    result = await call_tool(
        "convert_currency",
        {"from_currency": "USD", "to_currency": "INR", "amount": 100},
    )
    print(f"  Result: {result}")
except Exception as e:
    print(f"  Error: {e}")

7.5 — convert_currency(USD → INR, amount=100)
----------------------------------------
  Error: unhandled errors in a TaskGroup (1 sub-exception)


---
## 8. Client Configuration Snippets

Here are ready-to-use configs for popular MCP clients. Replace `{server-slug}` with your server's slug.

### Authentication options

| Method | Best for | Setup |
|--------|----------|-------|
| **OAuth** (recommended) | Interactive use in Claude/Cursor | Just the URL, no keys |
| **API Key** | Programmatic access, CI/CD | `x-portkey-api-key` header |

In [13]:
SERVER_SLUG = MCP_SERVER_SLUG  # or e.g. "linear", "github", "slack"

print("="*60)
print("  Claude Desktop Config")
print("="*60)
print(f"""
File: ~/Library/Application Support/Claude/claude_desktop_config.json

Option A — OAuth (recommended, no API key needed):
{json.dumps({
    "mcpServers": {
        SERVER_SLUG: {
            "url": f"{MCP_GATEWAY_URL}/{SERVER_SLUG}/mcp"
        }
    }
}, indent=2)}

Option B — API Key:
{json.dumps({
    "mcpServers": {
        SERVER_SLUG: {
            "url": f"{MCP_GATEWAY_URL}/{SERVER_SLUG}/mcp",
            "headers": {
                "x-portkey-api-key": "YOUR_PORTKEY_API_KEY"
            }
        }
    }
}, indent=2)}
""")

print("="*60)
print("  Cursor Config")
print("="*60)
print(f"""
Open: Cmd+Shift+P > Cursor Settings: Open MCP Settings

{json.dumps({
    "mcpServers": {
        SERVER_SLUG: {
            "url": f"{MCP_GATEWAY_URL}/{SERVER_SLUG}/mcp",
            "headers": {
                "x-portkey-api-key": "YOUR_PORTKEY_API_KEY"
            }
        }
    }
}, indent=2)}
""")

print("="*60)
print("  VS Code Config (with GitHub Copilot)")
print("="*60)
print(f"""
{json.dumps({
    "mcp.servers": {
        SERVER_SLUG: {
            "url": f"{MCP_GATEWAY_URL}/{SERVER_SLUG}/mcp",
            "headers": {
                "x-portkey-api-key": "YOUR_PORTKEY_API_KEY"
            }
        }
    }
}, indent=2)}
""")

  Claude Desktop Config

File: ~/Library/Application Support/Claude/claude_desktop_config.json

Option A — OAuth (recommended, no API key needed):
{
  "mcpServers": {
    "demo-mcp-server": {
      "url": "https://mcp.portkey.ai/demo-mcp-server/mcp"
    }
  }
}

Option B — API Key:
{
  "mcpServers": {
    "demo-mcp-server": {
      "url": "https://mcp.portkey.ai/demo-mcp-server/mcp",
      "headers": {
        "x-portkey-api-key": "YOUR_PORTKEY_API_KEY"
      }
    }
  }
}

  Cursor Config

Open: Cmd+Shift+P > Cursor Settings: Open MCP Settings

{
  "mcpServers": {
    "demo-mcp-server": {
      "url": "https://mcp.portkey.ai/demo-mcp-server/mcp",
      "headers": {
        "x-portkey-api-key": "YOUR_PORTKEY_API_KEY"
      }
    }
  }
}

  VS Code Config (with GitHub Copilot)

{
  "mcp.servers": {
    "demo-mcp-server": {
      "url": "https://mcp.portkey.ai/demo-mcp-server/mcp",
      "headers": {
        "x-portkey-api-key": "YOUR_PORTKEY_API_KEY"
      }
    }
  }
}



---
## 9. Using with AI Gateway (LLM + MCP)

The real power: combine Portkey's **AI Gateway** (for LLM calls) with the **MCP Gateway** (for tool calls) in a single agent flow.

```
Agent Loop:
  1. User message  ──▶  AI Gateway  ──▶  LLM (GPT-4, Claude, etc.)
  2. LLM says "call tool X"
  3. Tool call     ──▶  MCP Gateway ──▶  MCP Server
  4. Tool result   ──▶  AI Gateway  ──▶  LLM (with tool result)
  5. Final response to user
```

Both flows use the **same API key** and appear in the **same dashboard** — unified traces across LLM and tool calls.

In [14]:
from portkey_ai import Portkey

# ----- Configure your virtual key and model -----
VIRTUAL_KEY_SLUG = "@your-virtual-key"  # e.g. "@my-openai"
MODEL = f"{VIRTUAL_KEY_SLUG}/gpt-4o-mini"
# -------------------------------------------------

portkey = Portkey(api_key=PORTKEY_API_KEY)

# Step 1: Get available tools from MCP server
async def get_mcp_tools_as_openai_functions():
    """Fetch MCP tools and convert to OpenAI function calling format."""
    async with streamablehttp_client(
        GATEWAY_CONNECTION_URL, headers=GATEWAY_HEADERS
    ) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()

            openai_tools = []
            for tool in tools.tools:
                openai_tools.append({
                    "type": "function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description or "",
                        "parameters": tool.inputSchema or {"type": "object", "properties": {}},
                    },
                })
            return openai_tools

try:
    openai_tools = await get_mcp_tools_as_openai_functions()
    print(f"Converted {len(openai_tools)} MCP tools to OpenAI function format:\n")
    for t in openai_tools:
        print(f"  - {t['function']['name']}: {t['function']['description'][:60]}")
except Exception as e:
    print(f"Error fetching tools: {e}")
    openai_tools = []

Error fetching tools: unhandled errors in a TaskGroup (1 sub-exception)


In [15]:
# Step 2: Agent loop — LLM decides which tools to call

async def agent_with_mcp_tools(user_message: str):
    """Run an agent loop: LLM → tool call → LLM → response."""
    messages = [{"role": "user", "content": user_message}]

    print(f"User: {user_message}")
    print("-" * 60)

    # Call LLM with tools via AI Gateway
    response = portkey.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=openai_tools,
        tool_choice="auto",
        max_tokens=500,
    )

    assistant_msg = response.choices[0].message

    if not assistant_msg.tool_calls:
        print(f"Assistant: {assistant_msg.content}")
        return

    # Process tool calls via MCP Gateway
    messages.append(assistant_msg)

    for tool_call in assistant_msg.tool_calls:
        fn_name = tool_call.function.name
        fn_args = json.loads(tool_call.function.arguments)
        print(f"  Tool call: {fn_name}({fn_args})")

        result = await call_tool(fn_name, fn_args)
        result_text = str(result)
        print(f"  Tool result: {result_text[:200]}")

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result_text,
        })

    # Send tool results back to LLM
    final_response = portkey.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=500,
    )

    print(f"\nAssistant: {final_response.choices[0].message.content}")


# Run the agent
try:
    await agent_with_mcp_tools("What is the current time and what is 42 + 58?")
except Exception as e:
    print(f"Error: {e}")

User: What is the current time and what is 42 + 58?
------------------------------------------------------------
Error: Error code: 400 - {'status': 'failure', 'message': 'Following keys are not valid: your-virtual-key'}


In [16]:
# Another example: currency conversion
try:
    await agent_with_mcp_tools("Convert 500 USD to EUR and also to INR.")
except Exception as e:
    print(f"Error: {e}")

User: Convert 500 USD to EUR and also to INR.
------------------------------------------------------------
Error: Error code: 400 - {'status': 'failure', 'message': 'Following keys are not valid: your-virtual-key'}


---
## 10. Access Control & Tool Provisioning

Portkey's MCP Gateway gives platform teams fine-grained control:

### Workspace-level access
- Each MCP server is provisioned to specific **workspaces**
- Users in a workspace see only the servers they have access to
- API keys are scoped to workspaces

### Tool-level provisioning
- **Enable/disable specific tools** per workspace or user
- Disabled tools are filtered out of `tools/list` responses
- Calls to disabled tools return an error

### Authentication methods

| Auth Type | How it works | Use case |
|-----------|-------------|----------|
| `none` | No upstream auth needed | Public servers (DeepWiki, etc.) |
| `headers` | Gateway injects static headers | API-key-based servers |
| `oauth` | Per-user OAuth consent flow | GitHub, Slack, Linear |
| `oauth_client_credentials` | Machine-to-machine OAuth | Backend services |

### Rate limiting
```json
{
  "tools": {
    "rateLimit": { "requests": 100, "window": 60 },
    "blocked": ["dangerous_tool"],
    "allowed": ["safe_tool_1", "safe_tool_2"]
  }
}
```

All of this is configured in the Portkey dashboard under **MCP > Server Settings**.

---
## 11. Observability

Every request through the MCP Gateway is **fully logged**. You get:

| What's logged | Details |
|---|---|
| **User identity** | Who made the request (API key or OAuth user) |
| **Server & tool** | Which MCP server and which tool was called |
| **Parameters** | Full input arguments to the tool |
| **Response** | Complete tool response |
| **Latency** | Time to proxy to upstream and get response |
| **Errors** | Failed tool calls with error details |

### Viewing logs

1. Go to [app.portkey.ai](https://app.portkey.ai) → **MCP** in the sidebar
2. Click on a server to see its **Logs** tab
3. Filter by user, tool name, time range, or status

### Unified with AI Gateway

If you use both AI Gateway (for LLMs) and MCP Gateway (for tools):
- **Same dashboard** for both
- **Traces** span LLM calls and tool calls
- **Single API key** for everything

Let's make a few calls and then you can check the dashboard:

In [17]:
import time

print("Making tool calls for observability demo...")
print("=" * 60)

test_calls = [
    ("add_numbers", {"a": 100, "b": 200}),
    ("get_current_time", {}),
    ("generate_random_number", {"min_val": 1, "max_val": 50}),
    ("get_exchange_rates", {"base_currency": "EUR"}),
    ("convert_currency", {"from_currency": "GBP", "to_currency": "JPY", "amount": 250}),
]

for tool_name, args in test_calls:
    try:
        t0 = time.time()
        result = await call_tool(tool_name, args)
        elapsed = time.time() - t0
        
        result_str = str(result)
        if len(result_str) > 100:
            result_str = result_str[:100] + "..."
        
        print(f"  {tool_name:30s} [{elapsed:.2f}s] → {result_str}")
    except Exception as e:
        print(f"  {tool_name:30s} [ERROR] → {str(e)[:80]}")

print()
print("Check these calls in the Portkey dashboard:")
print("  https://app.portkey.ai → MCP → Logs")

Making tool calls for observability demo...
  add_numbers                    [ERROR] → unhandled errors in a TaskGroup (1 sub-exception)
  get_current_time               [ERROR] → unhandled errors in a TaskGroup (1 sub-exception)
  generate_random_number         [ERROR] → unhandled errors in a TaskGroup (1 sub-exception)
  get_exchange_rates             [ERROR] → unhandled errors in a TaskGroup (1 sub-exception)
  convert_currency               [ERROR] → unhandled errors in a TaskGroup (1 sub-exception)

Check these calls in the Portkey dashboard:
  https://app.portkey.ai → MCP → Logs


---
## 12. Cleanup

Delete the test MCP integration if you no longer need it.

In [18]:
# Uncomment to delete the demo integration

# INTEGRATION_ID = "<paste-integration-id-from-step-3>"  # from registration
#
# try:
#     resp = requests.delete(
#         f"{PORTKEY_BASE_URL}/mcp-integrations/{INTEGRATION_ID}",
#         headers={"x-portkey-api-key": PORTKEY_API_KEY},
#         timeout=30,
#     )
#     print(f"Status: {resp.status_code}")
#     print(f"Response: {resp.json()}")
#     if resp.status_code in (200, 204):
#         print("\nIntegration deleted successfully.")
# except Exception as e:
#     print(f"Error: {e}")

---
## Recap

The Portkey MCP Gateway gives you:

| Capability | Without Gateway | With Gateway |
|---|---|---|
| **Auth** | Each agent manages its own credentials for every server | Agents authenticate once to Portkey |
| **Access control** | No centralized control | Workspace + tool-level provisioning |
| **Credential management** | Keys scattered across agents | Gateway injects credentials per server |
| **Observability** | No visibility into tool usage | Full logs: who, what, when, parameters |
| **OAuth** | Each agent implements OAuth flows | Gateway handles OAuth + token refresh |
| **Rate limiting** | Per-server implementation | Centralized, configurable |

### Architecture

```
┌─────────────┐  ┌─────────────┐  ┌─────────────┐
│   Claude     │  │   Cursor    │  │  Custom Bot │
│   Desktop    │  │   IDE       │  │  (Python)   │
└──────┬───────┘  └──────┬──────┘  └──────┬──────┘
       │                 │                 │
       └────────┬────────┘─────────┬───────┘
                │   API Key / OAuth │
                ▼                   ▼
       ┌────────────────────────────────────┐
       │      Portkey MCP Gateway           │
       │  • Auth & access control           │
       │  • Credential injection            │
       │  • Tool provisioning               │
       │  • Logging & observability         │
       │  • Rate limiting                   │
       └────┬──────────┬──────────┬─────────┘
            │          │          │
            ▼          ▼          ▼
       ┌────────┐ ┌────────┐ ┌────────┐
       │ Linear │ │ GitHub │ │ Custom │
       │  MCP   │ │  MCP   │ │  MCP   │
       └────────┘ └────────┘ └────────┘
```

### Next steps

1. **Add servers:** [MCP Registry](https://app.portkey.ai) → Org Settings → MCP Registry
2. **Connect agents:** Use the client configs from Section 8
3. **Monitor:** MCP tab in the workspace sidebar for logs
4. **Docs:** [docs.portkey.ai/docs/product/mcp-gateway](https://docs.portkey.ai/docs/product/mcp-gateway)
5. **Self-host:** [github.com/Portkey-AI/gateway](https://github.com/Portkey-AI/gateway)